<a href="https://colab.research.google.com/github/Guilherme5G/APEX-f1-strategy-predictor/blob/main/nemecTesteHipotese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

df_espera = pd.read_csv('Ex4_TempoEspera.csv')
df_estresse = pd.read_csv('Ex3_Estresse.csv')
df_antes_depois = pd.read_csv('Ex2_AntesDepois.csv')

In [3]:
#1 tempo espera
dados_espera = df_espera.iloc[:, 0].dropna()

stat_shapiro, p_shapiro = stats.shapiro(dados_espera)
print(f"Shapiro-Wilk p-valor: {p_shapiro:.5f}")

alpha = 0.05

if p_shapiro >= alpha:
    print("-> Dados normais: aplicando Teste t de 1 amostra.")
    stat, p_valor = stats.ttest_1samp(dados_espera, popmean=15)
else:
    print("-> Dados não-normais: aplicando Teste de Wilcoxon (benchmark).")
    diferenca = dados_espera - 15
    stat, p_valor = stats.wilcoxon(diferenca)

print(f"Estatística: {stat:.4f} | p-valor: {p_valor:.5f}")

if p_valor < alpha:
    print("Decisão: Rejeitamos H0. O tempo médio difere significativamente de 15 minutos.")
else:
    print("Decisão: Não rejeitamos H0. Não há evidências para refutar a afirmação da empresa.")

Shapiro-Wilk p-valor: 0.00101
-> Dados não-normais: aplicando Teste de Wilcoxon (benchmark).
Estatística: 142.0000 | p-valor: 0.00017
Decisão: Rejeitamos H0. O tempo médio difere significativamente de 15 minutos.


In [7]:
print("Colunas encontradas:", df_estresse.columns.tolist())

if 'TI' in df_estresse.columns and 'RH' in df_estresse.columns:
    ti = df_estresse['TI'].dropna()
    rh = df_estresse['RH'].dropna()
else:
    col_depto = df_estresse.columns[0]
    col_valor = df_estresse.columns[1]
    ti = df_estresse[df_estresse[col_depto] == 'TI'][col_valor].dropna()
    rh = df_estresse[df_estresse[col_depto] == 'RH'][col_valor].dropna()

_, p_shapiro_ti = stats.shapiro(ti)
_, p_shapiro_rh = stats.shapiro(rh)
print(f"Shapiro TI p-valor: {p_shapiro_ti:.5f} | Shapiro RH p-valor: {p_shapiro_rh:.5f}")

alpha = 0.05
ambos_normais = (p_shapiro_ti >= alpha) and (p_shapiro_rh >= alpha)

if ambos_normais:
    _, p_levene = stats.levene(ti, rh)
    print(f"Levene p-valor: {p_levene:.5f}")

    if p_levene >= alpha:
        print("-> Dados normais e variâncias iguais: Teste t independente padrão.")
        stat, p_valor = stats.ttest_ind(ti, rh, equal_var=True)
    else:
        print("-> Dados normais e variâncias desiguais: Teste t de Welch.")
        stat, p_valor = stats.ttest_ind(ti, rh, equal_var=False)
else:
    print("-> Pelo menos um grupo não é normal: Teste de Mann-Whitney U.")
    stat, p_valor = stats.mannwhitneyu(ti, rh, alternative='two-sided')

print(f"Estatística: {stat:.4f} | p-valor: {p_valor:.5f}")

if p_valor < alpha:
    print("Decisão: Rejeitamos H0. Os departamentos possuem níveis de estresse significativamente diferentes.")
else:
    print("Decisão: Não rejeitamos H0. Não há diferença estatisticamente comprovada entre os departamentos.")


print(f"Média de Estresse TI: {ti.mean():.2f}")
print(f"Média de Estresse RH: {rh.mean():.2f}")

Colunas encontradas: ['Setor', 'Estresse']
Shapiro TI p-valor: 0.25509 | Shapiro RH p-valor: 0.71070
Levene p-valor: 0.42137
-> Dados normais e variâncias iguais: Teste t independente padrão.
Estatística: 2.4688 | p-valor: 0.01608
Decisão: Rejeitamos H0. Os departamentos possuem níveis de estresse significativamente diferentes.
Média de Estresse TI: 59.69
Média de Estresse RH: 54.88


In [6]:
antes = df_antes_depois.iloc[:, 0].dropna()
depois = df_antes_depois.iloc[:, 1].dropna()

diferencas = depois - antes
stat_shapiro, p_shapiro = stats.shapiro(diferencas)
print(f"Shapiro-Wilk (diferenças) p-valor: {p_shapiro:.5f}")

alpha = 0.05

if p_shapiro >= alpha:
    print("-> Diferenças normais: aplicando Teste t pareado (ttest_rel).")
    stat, p_valor = stats.ttest_rel(depois, antes, alternative='greater')
else:
    print("-> Diferenças não-normais: aplicando Teste de Wilcoxon pareado.")
    stat, p_valor = stats.wilcoxon(depois, antes, alternative='greater')

print(f"Estatística: {stat:.4f} | p-valor: {p_valor:.5f}")

if p_valor < alpha:
    print("Decisão: Rejeitamos H0. O curso teve um impacto positivo significativo no desempenho.")
else:
    print("Decisão: Não rejeitamos H0. Não há evidências estatísticas de que o curso melhorou as notas.")

Shapiro-Wilk (diferenças) p-valor: 0.25042
-> Diferenças normais: aplicando Teste t pareado (ttest_rel).
Estatística: 23.6772 | p-valor: 0.00000
Decisão: Rejeitamos H0. O curso teve um impacto positivo significativo no desempenho.
